# Phase 8 — Simulation Engine (corrected billing)

**Purpose:** Answer "if ESCoSA approves a price rise, when do SA2s cross affordability thresholds?"

**Inputs:**
- `data/clean/clean_master_sa2_v2.csv` — SA2s with corrected FY2024-25 bills and WPI-adjusted income
- `data/clean/clean_master_sa2_v2.gpkg` — same with geometry

**Billing approach:**
- Simulations apply a usage rate multiplier to `bill_usage_only` (tiered usage component)
- Fixed supply charge ($314.40) and sewerage are held constant — standard regulatory assumption
- FY2025-26 modelled with updated water usage rates and supply charge only (sewerage rate TBC for 2025-26)

**Story change from corrected tariffs:**
- Old (incorrect) bill: $2,248/yr → 10 Critical SA2s at baseline
- Corrected bill: ~$1,160–$1,344/yr → 0 Critical, 0 High SA2s at baseline
- Simulation now shows when suburbs first cross the High (3%) and Critical (4%) thresholds

**Outputs:**
- `outputs/simulation/scenario_results.csv` — per-SA2 per-scenario results
- `outputs/simulation/tipping_points.csv` — % rise at which each SA2 crosses next tier
- `outputs/simulation/scenario_tier_counts.html`
- `outputs/simulation/tier_shift_map_10pct.html`
- `outputs/simulation/interactive_scenario_map.html`

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
import plotly.express as px
import json
import numpy as np

ROOT      = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
DATA_CLEAN = ROOT / 'data' / 'clean'
SIM_OUT   = ROOT / 'outputs' / 'simulation'
FIG_OUT   = ROOT / 'outputs' / 'figures'

SIM_OUT.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)

Root: C:\Users\mussa\OneDrive\Desktop\Projects\sa_water_project


## 1. Tariff constants and simulation parameters

In [2]:
# FY2024-25 fixed components (not scaled in usage-rate simulations)
SUPPLY_2425   = 314.40   # $/year fixed supply charge
SEWER_METRO   = 0.622    # $/year per $1K property value
SEWER_COUNTRY = 0.928    # $/year per $1K property value
PROP_VALUE_K  = 600.0    # $600K baseline property value

# FY2025-26 rates (for the exact 2025-26 scenario)
TIER1_RATE_2526 = 2.357
TIER2_RATE_2526 = 3.365
TIER3_RATE_2526 = 3.646
SUPPLY_2526     = 329.20
TIER1_KL        = 140.0
TIER2_KL        = 520.0
TYPICAL_KL      = 189.0

# Absolute tier thresholds
TIER_THRESHOLDS = {'Critical': 0.04, 'High': 0.03, 'Moderate': 0.02}
TIER_ORDER   = ['Critical', 'High', 'Moderate', 'Low', 'Unknown']
TIER_COLOURS = {
    'Critical': '#d62728',
    'High':     '#ff7f0e',
    'Moderate': '#ffdf00',
    'Low':      '#2ca02c',
    'Unknown':  '#aec7e8',
}

def usage_2526(usage_kl):
    t1 = min(usage_kl, TIER1_KL) * TIER1_RATE_2526
    t2 = min(max(0, usage_kl - TIER1_KL), TIER2_KL - TIER1_KL) * TIER2_RATE_2526
    t3 = max(0, usage_kl - TIER2_KL) * TIER3_RATE_2526
    return t1 + t2 + t3

usage_2526_typical = usage_2526(TYPICAL_KL)
print(f'FY2025-26 usage component (189 kL): ${usage_2526_typical:.2f}')
print(f'FY2024-25 usage component (189 kL): $472.63')
print(f'Usage % increase 2024-25 → 2025-26: {(usage_2526_typical/472.63 - 1)*100:.1f}%')
print(f'Supply increase: ${SUPPLY_2526 - SUPPLY_2425:.2f}/yr ({(SUPPLY_2526/SUPPLY_2425-1)*100:.1f}%)')

FY2025-26 usage component (189 kL): $494.87
FY2024-25 usage component (189 kL): $472.63
Usage % increase 2024-25 → 2025-26: 4.7%
Supply increase: $14.80/yr (4.7%)


## 2. Load data

In [3]:
df = pd.read_csv(DATA_CLEAN / 'clean_master_sa2_v2.csv', dtype={'SA2_CODE21': str})
gdf = gpd.read_file(DATA_CLEAN / 'clean_master_sa2_v2.gpkg')
gdf['SA2_CODE21'] = gdf['SA2_CODE21'].astype(str)
gdf = gdf.to_crs('EPSG:4326')

print('CSV shape:', df.shape)
print('CRS:', gdf.crs)
print()
print('Baseline absolute tier counts (corrected FY2024-25 tariff):')
print(df['burden_tier_abs'].value_counts().reindex(TIER_ORDER, fill_value=0))
print()
print('Corrected bill range (SA Water only):')
saw = df[df['provider_type'] == 'SA Water']
print(f'  Metro:   ${saw[saw["is_metro"]]["bill_owner_blended"].iloc[0]:.2f} (uniform)')
print(f'  Country: ${saw[~saw["is_metro"]]["bill_owner_blended"].iloc[0]:.2f} (uniform)')

CSV shape: (176, 37)
CRS: EPSG:4326

Baseline absolute tier counts (corrected FY2024-25 tariff):
burden_tier_abs
Critical      0
High          0
Moderate     24
Low         144
Unknown       8
Name: count, dtype: int64

Corrected bill range (SA Water only):
  Metro:   $1160.23 (uniform)
  Country: $1343.83 (uniform)


## 3. Simulation function

The simulation scales **usage charges only** — supply ($314.40) and sewerage are held
constant, consistent with them being separate regulatory determinations.

`bill_usage_only` from Phase 5 is the pre-computed tiered usage at baseline rates.
Applying a multiplier scales all three tiers proportionally (same % change to T1/T2/T3).

In [4]:
def assign_abs_tier(ratio):
    if pd.isna(ratio):               return 'Unknown'
    if ratio > TIER_THRESHOLDS['Critical']: return 'Critical'
    if ratio > TIER_THRESHOLDS['High']:     return 'High'
    if ratio > TIER_THRESHOLDS['Moderate']: return 'Moderate'
    return 'Low'


def run_scenario(
    df_in,
    price_increase_pct,
    income_growth_pct=0.0,
    supply_override=None,
):
    """
    Recalculate sim_annual_bill and sim_burden_ratio for every SA Water SA2.
    price_increase_pct: % applied to usage rates (Tier 1/2/3 proportionally)
    supply_override:    if set, replaces SUPPLY_2425 (used for FY2025-26 scenario)
    """
    out = df_in.copy()
    usage_mult   = 1 + price_increase_pct / 100
    supply_fixed = supply_override if supply_override is not None else SUPPLY_2425

    # Fixed non-usage component (supply + sewerage) is unchanged
    fixed = out['bill_owner_blended'] - out['bill_usage_only']
    # Replace supply component if overridden
    if supply_override is not None:
        fixed = fixed + (supply_override - SUPPLY_2425)

    out['sim_annual_bill']   = (out['bill_usage_only'] * usage_mult + fixed).round(2)

    # Non-SA Water SA2s: leave bill as NaN
    non_saw = out['provider_type'] != 'SA Water'
    out.loc[non_saw, 'sim_annual_bill'] = float('nan')

    adj_income = out['median_hhd_inc_annual_adj'] * (1 + income_growth_pct / 100)
    out['sim_burden_ratio']  = (out['sim_annual_bill'] / adj_income).round(6)
    out['sim_tier_abs']      = out['sim_burden_ratio'].apply(assign_abs_tier)

    # Tier shift vs baseline (0 = no change, +1 = one tier worse)
    tier_rank = {'Low': 0, 'Moderate': 1, 'High': 2, 'Critical': 3, 'Unknown': None}
    out['baseline_rank'] = out['burden_tier_abs'].map(tier_rank)
    out['sim_rank']      = out['sim_tier_abs'].map(tier_rank)
    out['tier_shift']    = out['sim_rank'] - out['baseline_rank']

    return out


# Validate: 0% rise must reproduce Phase 5 absolute tiers exactly
ref = run_scenario(df, 0)
match = (ref['sim_tier_abs'] == ref['burden_tier_abs']).all()
print(f'Reference (0% rise) matches Phase 5 exactly: {match}')
print(ref['sim_tier_abs'].value_counts().reindex(TIER_ORDER, fill_value=0))

Reference (0% rise) matches Phase 5 exactly: True
sim_tier_abs
Critical      0
High          0
Moderate     24
Low         144
Unknown       8
Name: count, dtype: int64


## 4. Run scenarios

In [5]:
# FY2024-25 usage-rate scenarios
PCT_SCENARIOS = [0, 5, 10, 15, 20, 25, 30]

# FY2025-26 exact rates (~+4.7% usage + $14.80 supply)
# Compute effective % increase in usage from 2024-25 to 2025-26 at 189kL
usage_2425 = 472.63
usage_pct_2526 = (usage_2526_typical / usage_2425 - 1) * 100

results = {}
for pct in PCT_SCENARIOS:
    results[pct] = run_scenario(df, pct)

# 2025-26 as a named scenario
results['2025-26'] = run_scenario(df, usage_pct_2526, supply_override=SUPPLY_2526)

print('Scenarios computed:', PCT_SCENARIOS + ['2025-26'])

Scenarios computed: [0, 5, 10, 15, 20, 25, 30, '2025-26']


In [ ]:
def bill_label(pct, supply=SUPPLY_2425):
    """Representative metro owner-bill for a scenario."""
    usage = usage_2425 * (1 + pct/100)
    return f'${usage + supply + PROP_VALUE_K * SEWER_METRO:.2f}'

rows = []
for key, r in results.items():
    known = r[(r['sim_tier_abs'] != 'Unknown') & (r['provider_type'] == 'SA Water')]
    base  = r[r['burden_tier_abs'] != 'Unknown']

    new_high     = int(((known['sim_tier_abs'] == 'High')     & (known['burden_tier_abs'] == 'Moderate')).sum())
    new_critical = int(((known['sim_tier_abs'] == 'Critical') & (known['burden_tier_abs'] != 'Critical')).sum())
    total_shifts = int((known['tier_shift'].abs() > 0).sum())

    if key == '2025-26':
        label = 'FY2025-26 water + supply only'
        supply = SUPPLY_2526
        pct_val = usage_pct_2526
    else:
        label = f'+{key}% (usage rates)' if key > 0 else 'Baseline (FY2024-25)'
        supply = SUPPLY_2425
        pct_val = key

    rows.append({
        'Scenario':      label,
        'Metro bill ($)': bill_label(pct_val, supply),
        'Critical':      int((known['sim_tier_abs'] == 'Critical').sum()),
        'High':          int((known['sim_tier_abs'] == 'High').sum()),
        'Moderate':      int((known['sim_tier_abs'] == 'Moderate').sum()),
        'Low':           int((known['sim_tier_abs'] == 'Low').sum()),
        'New High':      new_high,
        'New Critical':  new_critical,
        'Total Shifts':  total_shifts,
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 5. Tipping point analysis

For each SA2, compute the exact % usage-rate increase at which it crosses the next tier boundary.
Formula: `tipping_pct = (income × threshold - fixed_bill) / bill_usage_only × 100 - 100`

In [7]:
saw = df[(df['provider_type'] == 'SA Water') & df['water_cost_burden_ratio'].notna()].copy()

next_threshold = {
    'Low':      TIER_THRESHOLDS['Moderate'],
    'Moderate': TIER_THRESHOLDS['High'],
    'High':     TIER_THRESHOLDS['Critical'],
}

# fixed_bill = supply + sewerage (held constant in usage-rate simulations)
saw['fixed_bill'] = saw['bill_owner_blended'] - saw['bill_usage_only']

def calc_tipping(row):
    tier = row['burden_tier_abs']
    if tier not in next_threshold or row['bill_usage_only'] <= 0 or pd.isna(row['median_hhd_inc_annual_adj']):
        return float('nan')
    threshold = next_threshold[tier]
    needed_bill  = row['median_hhd_inc_annual_adj'] * threshold
    needed_usage = needed_bill - row['fixed_bill']
    if needed_usage <= 0:
        return 0.0
    return (needed_usage / row['bill_usage_only'] - 1) * 100

saw['tipping_pct'] = saw.apply(calc_tipping, axis=1)
saw['next_tier']   = saw['burden_tier_abs'].map({
    'Low': 'Moderate', 'Moderate': 'High', 'High': 'Critical'
})

tipping_out = saw[saw['tipping_pct'].notna()].sort_values('tipping_pct').head(30)[[
    'SA2_NAME21', 'SA3_NAME21', 'is_metro',
    'burden_tier_abs', 'next_tier',
    'water_cost_burden_ratio', 'tipping_pct',
    'median_hhd_inc_annual_adj',
]].copy()
tipping_out['current_pct'] = (tipping_out['water_cost_burden_ratio'] * 100).round(2)
tipping_out['tipping_pct'] = tipping_out['tipping_pct'].round(1)

print('Top 30 SA2s closest to crossing next tier (lowest tipping-point % rise):')
print(tipping_out[['SA2_NAME21', 'SA3_NAME21', 'burden_tier_abs', 'next_tier',
                    'current_pct', 'tipping_pct']].to_string(index=False))

Top 30 SA2s closest to crossing next tier (lowest tipping-point % rise):
                      SA2_NAME21                    SA3_NAME21 burden_tier_abs next_tier  current_pct  tipping_pct
            Port Pirie Surrounds                     Mid North             Low  Moderate         1.98          3.5
                       Jamestown                     Mid North             Low  Moderate         1.98          3.5
                 West Coast (SA) Eyre Peninsula and South West             Low  Moderate         1.97          4.3
                  Christie Downs                   Onkaparinga             Low  Moderate         1.96          4.7
                          Kadina               Yorke Peninsula             Low  Moderate         1.96          6.0
Kimba - Cleve - Franklin Harbour Eyre Peninsula and South West             Low  Moderate         1.92         11.3
                         Whyalla Eyre Peninsula and South West             Low  Moderate         1.90         14.3
       

In [8]:
# Save full tipping-point table
tipping_full = saw[saw['tipping_pct'].notna()].sort_values('tipping_pct')[[
    'SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21', 'is_metro',
    'burden_tier_abs', 'next_tier', 'water_cost_burden_ratio',
    'median_hhd_inc_annual_adj', 'bill_usage_only', 'bill_owner_blended',
    'tipping_pct'
]].copy()
tipping_full['current_burden_pct'] = (tipping_full['water_cost_burden_ratio'] * 100).round(2)
tipping_full.to_csv(SIM_OUT / 'tipping_points.csv', index=False)
print(f'Saved tipping_points.csv ({len(tipping_full)} rows)')

Saved tipping_points.csv (168 rows)


## 6. SA2s newly entering High/Critical under each scenario

In [ ]:
print('SA2s newly entering High or Critical tier by scenario:')
print()
for key in PCT_SCENARIOS[1:] + ['2025-26']:
    r = results[key]
    saw_r = r[r['provider_type'] == 'SA Water']
    entered_high = saw_r[
        (saw_r['sim_tier_abs'] == 'High') &
        (saw_r['burden_tier_abs'] == 'Moderate')
    ]
    entered_crit = saw_r[
        (saw_r['sim_tier_abs'] == 'Critical') &
        (saw_r['burden_tier_abs'] != 'Critical')
    ]
    label = 'FY2025-26 water + supply only' if key == '2025-26' else f'+{key}%'
    print(f'{label}: {len(entered_high)} new High, {len(entered_crit)} new Critical')
    if len(entered_high) > 0 or len(entered_crit) > 0:
        new_entrants = pd.concat([entered_high, entered_crit])
        print(new_entrants[['SA2_NAME21', 'burden_tier_abs', 'sim_tier_abs',
                             'sim_burden_ratio', 'median_hhd_inc_annual_adj']]
              .sort_values('sim_burden_ratio', ascending=False)
              .to_string(index=False))
    print()

## 7. Charts

In [10]:
# Tier count bar chart
tiers_to_plot = ['Critical', 'High', 'Moderate', 'Low']
scenario_labels = summary['Scenario'].tolist()

fig_counts = go.Figure()
for tier in tiers_to_plot:
    fig_counts.add_trace(go.Bar(
        name=tier,
        x=scenario_labels,
        y=summary[tier],
        marker_color=TIER_COLOURS[tier],
        text=summary[tier],
        textposition='auto',
    ))

fig_counts.update_layout(
    title='SA2 Absolute Stress Tier Distribution — Price Rise Scenarios (corrected FY2024-25 tariff)',
    xaxis_title='Price Increase Scenario',
    yaxis_title='Number of SA2 Areas',
    barmode='group',
    legend_title='Stress Tier',
    template='plotly_white',
    height=500,
)
fig_counts.write_html(SIM_OUT / 'scenario_tier_counts.html')
print('Saved: scenario_tier_counts.html')

Saved: scenario_tier_counts.html


In [11]:
# Choropleth: tier under +10% scenario
r10 = results[10][['SA2_CODE21', 'sim_tier_abs', 'tier_shift',
                    'sim_burden_ratio', 'sim_annual_bill']].copy()
gdf10 = gdf.merge(r10, on='SA2_CODE21', how='left')

def shift_label(s):
    if pd.isna(s) or s == 0: return 'No change'
    return f'+{int(s)} tier' if s > 0 else f'{int(s)} tier'

gdf10['shift_label'] = gdf10['tier_shift'].apply(shift_label)
gdf10['sim_tier_abs'] = gdf10['sim_tier_abs'].fillna('Unknown')

geojson10 = json.loads(gdf10.to_json())

fig_10pct = px.choropleth_mapbox(
    gdf10,
    geojson=geojson10,
    locations=gdf10.index,
    color='sim_tier_abs',
    color_discrete_map=TIER_COLOURS,
    category_orders={'sim_tier_abs': TIER_ORDER},
    hover_name='SA2_NAME21',
    hover_data={
        'sim_tier_abs': True,
        'shift_label': True,
        'sim_burden_ratio': ':.2%',
        'sim_annual_bill': ':$.2f',
    },
    mapbox_style='carto-positron',
    center={'lat': -30.0, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='SA Water Stress Tiers After +10% Usage Rate Rise (corrected FY2024-25 baseline)',
)
fig_10pct.update_layout(height=700, margin={'r':0,'t':50,'l':0,'b':0})
fig_10pct.write_html(SIM_OUT / 'tier_shift_map_10pct.html')
print('Saved: tier_shift_map_10pct.html')

C:\Users\mussa\AppData\Local\Temp\ipykernel_58504\2331049107.py:15: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_10pct = px.choropleth_mapbox(


Saved: tier_shift_map_10pct.html


In [ ]:
# Interactive scenario map with dropdown
all_gdf = gdf[['SA2_CODE21', 'SA2_NAME21', 'geometry']].copy()
ALL_KEYS = PCT_SCENARIOS + ['2025-26']

for key in ALL_KEYS:
    r = results[key][['SA2_CODE21', 'sim_tier_abs', 'sim_burden_ratio', 'sim_annual_bill']]
    r = r.rename(columns={
        'sim_tier_abs':   f'tier_{key}',
        'sim_burden_ratio': f'ratio_{key}',
        'sim_annual_bill':  f'bill_{key}',
    })
    all_gdf = all_gdf.merge(r, on='SA2_CODE21', how='left')

geojson_all = json.loads(all_gdf.to_json())
idx = list(range(len(all_gdf)))

fig_interactive = go.Figure()
for i, key in enumerate(ALL_KEYS):
    tier_col  = f'tier_{key}'
    ratio_col = f'ratio_{key}'
    bill_col  = f'bill_{key}'
    label     = 'FY2025-26 water + supply only' if key == '2025-26' else (f'+{key}%' if key > 0 else 'Baseline')

    tier_vals = all_gdf[tier_col].fillna('Unknown')
    tier_nums = tier_vals.map({'Low':0,'Moderate':1,'High':2,'Critical':3,'Unknown':4})

    fig_interactive.add_trace(go.Choroplethmapbox(
        geojson=geojson_all,
        locations=idx,
        z=tier_nums,
        colorscale=[
            [0.00, TIER_COLOURS['Low']],
            [0.25, TIER_COLOURS['Low']],
            [0.25, TIER_COLOURS['Moderate']],
            [0.50, TIER_COLOURS['Moderate']],
            [0.50, TIER_COLOURS['High']],
            [0.75, TIER_COLOURS['High']],
            [0.75, TIER_COLOURS['Critical']],
            [1.00, TIER_COLOURS['Critical']],
        ],
        zmin=0, zmax=3,
        showscale=False,
        marker_opacity=0.75,
        marker_line_width=0.3,
        name=label,
        text=all_gdf['SA2_NAME21'] + '<br>' + tier_vals + '<br>' +
             (all_gdf[ratio_col] * 100).round(2).astype(str) + '% burden<br>' +
             '$' + all_gdf[bill_col].round(2).astype(str) + '/yr',
        hovertemplate='%{text}<extra></extra>',
        visible=(key == 0),
    ))

buttons = []
for i, key in enumerate(ALL_KEYS):
    lbl = 'FY2025-26 water + supply only' if key == '2025-26' else (f'+{key}% usage rise' if key > 0 else 'Baseline (FY2024-25)')
    buttons.append(dict(
        label=lbl,
        method='update',
        args=[{'visible': [j == i for j in range(len(ALL_KEYS))]},
              {'title': f'SA Water Stress Tiers — {lbl}'}]
    ))

fig_interactive.update_layout(
    title='SA Water Stress Tiers — Baseline (FY2024-25)',
    mapbox_style='carto-positron',
    mapbox_center={'lat': -30.0, 'lon': 135.5},
    mapbox_zoom=4.5,
    margin={'r':0,'t':60,'l':0,'b':0},
    height=750,
    updatemenus=[dict(
        active=0, buttons=buttons,
        direction='down',
        pad={'r':10,'t':10},
        showactive=True,
        x=0.01, xanchor='left',
        y=0.99, yanchor='top',
    )],
    annotations=[dict(
        text='Select scenario:',
        x=0.01, y=1.04,
        xref='paper', yref='paper',
        showarrow=False, align='left',
    )],
)
fig_interactive.write_html(SIM_OUT / 'interactive_scenario_map.html')
print('Saved: interactive_scenario_map.html')

## 8. Save scenario results CSV (for Power BI)

In [ ]:
all_rows = []
for key, r in results.items():
    subset = r[[
        'SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21',
        'provider_type', 'is_metro',
        'burden_tier_abs', 'burden_tier_rel',
        'median_hhd_inc_annual_adj',
        'sim_annual_bill', 'sim_burden_ratio', 'sim_tier_abs', 'tier_shift',
    ]].copy()
    if key == '2025-26':
        subset.insert(0, 'scenario_label', 'FY2025-26 water + supply only')
        subset.insert(0, 'price_increase_pct', round(usage_pct_2526, 1))
    else:
        subset.insert(0, 'scenario_label', f'+{key}%' if key > 0 else 'Baseline')
        subset.insert(0, 'price_increase_pct', key)
    all_rows.append(subset)

scenario_results = pd.concat(all_rows, ignore_index=True)
scenario_results['sim_burden_pct'] = (scenario_results['sim_burden_ratio'] * 100).round(4)
scenario_results['tier_shifted']   = (scenario_results['tier_shift'].abs() > 0).astype(int)

# Tier order integers for Power BI sorting
tier_rank = {'Critical': 1, 'High': 2, 'Moderate': 3, 'Low': 4, 'Unknown': 5}
scenario_results['sim_tier_order'] = scenario_results['sim_tier_abs'].map(tier_rank)

scenario_results.to_csv(SIM_OUT / 'scenario_results.csv', index=False)
print(f'Saved: scenario_results.csv  ({scenario_results.shape[0]} rows × {scenario_results.shape[1]} cols)')
print(scenario_results.head(3).to_string())

In [14]:
print('=== PHASE 8 COMPLETE ===')
print()
print('Key story change from corrected tariffs:')
print('  Old bill ($2,248): 10 Critical SA2s at FY2024-25 baseline')
print('  Corrected bill (~$1,160–$1,344): 0 Critical, 0 High at baseline')
print()
print('Simulation story:')
tp = tipping_out.head(5)
for _, row in tp.iterrows():
    print(f'  {row["SA2_NAME21"]}: crosses {row["next_tier"]} at +{row["tipping_pct"]:.0f}% usage rise')
print()
print('Outputs:')
for f in sorted(SIM_OUT.glob('*.csv')) + sorted(SIM_OUT.glob('*.html')):
    print(f'  {f.name}')
print()
print('Next: Phase 9 — SHAP explainability (re-run with new SEIFA-only model from Phase 6)')

=== PHASE 8 COMPLETE ===

Key story change from corrected tariffs:
  Old bill ($2,248): 10 Critical SA2s at FY2024-25 baseline
  Corrected bill (~$1,160–$1,344): 0 Critical, 0 High at baseline

Simulation story:
  Port Pirie Surrounds: crosses Moderate at +4% usage rise
  Jamestown: crosses Moderate at +4% usage rise
  West Coast (SA): crosses Moderate at +4% usage rise
  Christie Downs: crosses Moderate at +5% usage rise
  Kadina: crosses Moderate at +6% usage rise

Outputs:
  scenario_results.csv
  tipping_points.csv
  interactive_scenario_map.html
  scenario_tier_counts.html
  tier_shift_map_10pct.html
  tier_shift_map_7pct.html

Next: Phase 9 — SHAP explainability (re-run with new SEIFA-only model from Phase 6)
